In [4]:
import pandas as pd
import time

import matplotlib.pyplot as plt
import json
import numpy as np
from pathlib import Path

from langchain_community.callbacks import get_openai_callback
from langchain_classic.chains import create_retrieval_chain
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma
import re


from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import BSHTMLLoader

import hashlib
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path

import random
from pydantic import BaseModel, Field
from collections import defaultdict

random.seed(42) # I sure hope this works! 

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# ── UNC AI Gateway (Azure API Management in front of Azure OpenAI — OpenAI-compatible v1 API) ──
# One key, one endpoint, every course model. Your personal key arrives via Canvas Inbox.
# Create a .env file next to this notebook containing a single line:
# See .env.example for the required API key configuration.
# ⚠️  Never paste your key into a notebook cell — notebooks get submitted!
UNC_AI_BASE_URL = "https://azureaiapi.cloud.unc.edu/openai/v1"
UNC_AI_API_KEY = os.getenv("UNC_AI_API_KEY") or os.getenv("OPENAI_API_KEY")
assert UNC_AI_API_KEY, "Missing UNC_AI_API_KEY — check the Canvas Inbox message with your personal key"

# Course model lineup (deployment names on the gateway match the model names)
CHAT_MODEL = os.getenv("UNC_CHAT_MODEL", "gpt-4.1-mini")        # everyday workhorse
ADVANCED_MODEL = os.getenv("UNC_ADVANCED_MODEL", "gpt-5.6-sol")  # frontier tier for harder tasks
EMBED_MODEL = os.getenv("UNC_EMBED_MODEL", "text-embedding-3-small")

# Let every OpenAI-compatible library (LangChain, LlamaIndex, CrewAI, AutoGen…) find the gateway:
os.environ["OPENAI_API_KEY"] = UNC_AI_API_KEY
os.environ["OPENAI_BASE_URL"] = UNC_AI_BASE_URL

# Gateway policy: if you ever use the Responses API, set store=False and never background=True.

print("✓ UNC AI Gateway configured")
print(f"  Chat: {CHAT_MODEL}  |  Advanced: {ADVANCED_MODEL}  |  Embeddings: {EMBED_MODEL}")

✓ UNC AI Gateway configured
  Chat: gpt-4.1-mini  |  Advanced: gpt-5.6-sol  |  Embeddings: text-embedding-3-small


In [6]:
print(f"Gateway: {UNC_AI_BASE_URL}")
print(f"Embedding model: {EMBED_MODEL}")

# Embeddings through the UNC AI Gateway
try:
    embeddings = OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=UNC_AI_BASE_URL,
        api_key=UNC_AI_API_KEY,
    )

    # Test with a simple embedding
    print("Testing embedding with single text...")
    test_embedding = embeddings.embed_query("test")
    print(f"✓ Embedding successful! Vector dimension: {len(test_embedding)}")

except Exception as e:
    print(f"✗ Embedding failed: {e}")
    print("\nTroubleshooting:")
    print("1. Confirm UNC_AI_API_KEY is set in your .env file")
    print("2. Verify your key works: it was sent to you via Canvas Inbox")
    print("3. Check the model name — see the course model lineup in the setup cell")
    raise

print("✓ Embeddings client ready")

Gateway: https://azureaiapi.cloud.unc.edu/openai/v1
Embedding model: text-embedding-3-small
Testing embedding with single text...
✓ Embedding successful! Vector dimension: 1536
✓ Embeddings client ready


In [7]:
HEADERS = [("h1", "h1"), ("h2", "h2"), ("h3", "h3")]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=HEADERS)
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=150, add_start_index=True  # match your existing settings
)

def load_htm(path: str) -> list[Document]:
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    for tag in soup(["script", "style", "noscript", "iframe", "svg", "nav", "footer", "header", "form"]):
        tag.decompose()
    cleaned = str(soup)

    sections = html_splitter.split_text(cleaned)
    docs = []
    for i, s in enumerate(sections):
        section = " > ".join(s.metadata[k] for k in ("h1", "h2", "h3") if k in s.metadata)
        docs.append(Document(
            page_content=s.page_content,
            metadata={"source": path, "title": title,
                      "section": section or f"section_{i}", "section_idx": i},
        ))
    return docs

def add_chunk_ids(chunks: list[Document]) -> list[Document]:
    for c in chunks:
        m = c.metadata
        key = f"{m.get('source')}|{m.get('page')}|{m.get('row')}|{m.get('section_idx')}|{m.get('start_index')}"
        m["chunk_id"] = hashlib.sha1(key.encode()).hexdigest()[:12]
    return chunks

In [15]:
def load_document(file_path):
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        docs = PyPDFLoader(str(path)).load()      # metadata: source, page (0-indexed)
        ftype = "pdf"
        
    elif suffix == ".csv":
        docs = CSVLoader(str(path)).load()        # metadata: source, row
        ftype = "csv"
        
    elif suffix in (".htm", ".html"):
        docs = load_htm(str(path))                # metadata: source, title, section, section_idx
        ftype = "html"

    elif suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            article = json.load(f)

        docs = [
            Document(
                page_content=article["text"],
                metadata={
                    "source": str(path),
                    "title": article["title"],
                    "source_url": article["source"],
                    "revision_id": article["revision_id"],
                    "pageid": article["pageid"]
                }
            )
        ]
        ftype = "json"
        
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    for d in docs:
        d.metadata["file_type"] = ftype
        
    return docs

In [16]:
data_dir = Path("corpus/new_files")

files = list(data_dir.rglob("*.pdf"))
files += list(data_dir.rglob("*.htm"))
files += list(data_dir.rglob("*.csv"))
files += list(data_dir.rglob("*.json"))

print(f"Found {len(files)} files")

Found 71 files


In [17]:
all_documents = []

for file in files:
    docs = load_document(file)
    all_documents.extend(docs)

    print(f"{file.name}: {len(docs)} documents")

print(f"Total documents loaded: {len(all_documents)}")    

Advances_in_Molecular_Techniqu.pdf: 33 documents
Advances_in_semiochemicals_of_.pdf: 11 documents
Biochemical_Responses_of_Ataca.pdf: 22 documents
Biofortification_of_Sweetpotat.pdf: 30 documents
Biotic_Stress_Resistance_in_Sw.pdf: 33 documents
Can_Hyperspectral_Reflectance_.pdf: 20 documents
Changes_in_Chlorophyll_iai.pdf: 13 documents
Comparative_Transcriptome_Anal.pdf: 16 documents
Comparison_of_the_Effects_of_P.pdf: 22 documents
Development_of_a_transferrable.pdf: 8 documents
Effects_of_Biochar_Combined_wi.pdf: 22 documents
Effects_of_drying_on_the_bioac.pdf: 11 documents
Effect_of_fertilizer_on_bioche.pdf: 13 documents
Effect_of_Maize_(Zea_mays_L.)_.pdf: 11 documents
Effect_of_Paclobutrazol_on_Gro.pdf: 14 documents
Effect_of_Plant_Growth-Promoti.pdf: 26 documents
EFFECT_OF_POPULATION_DENSITY_O.pdf: 17 documents
Evaluation_of_Stem_Polyphenol_.pdf: 15 documents
Evaluation_of_Tuber_Yield_Perf.pdf: 9 documents
Farmers'_knowledge_and_practic.pdf: 15 documents
Genetic_Diversity_of_Sweet

In [18]:
# check to make sure this is actually what I wanted! 

# should be from a pdf
print(f"\nPreview ({len(all_documents[0].page_content)} characters):")
print(all_documents[0].page_content[:500] + "...")
print("\n")
# should be from a htm file
print(f"\nPreview ({len(all_documents[55].page_content)} characters):")
print(all_documents[8].page_content[:600] + "...")
print("\n")
# should be from a csv file
print(f"\nPreview ({len(all_documents[-1].page_content)} characters):")
print(all_documents[-1].page_content[:500] + "...")



Preview (3719 characters):
Academic Editor: Svetlana
Y. Folimonova
Received: 31 May 2026
Revised: 12 August 2026
Accepted: 14 August 2026
Published: 18 August 2026
Copyright: © 2026 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed under the terms and
conditions of the Creative Commons
Attribution (CC BY) license.
Review
Advances in Molecular Techniques for Detecting Sweet Potato
(Ipomoea batatas (L.) Lam) Viruses: A Comprehensive Review
Muhammad Abul Kalam Azad 1,*
 , N...



Preview (2039 characters):
Viruses 2026, 18, 908 9 of 32
5. Molecular Laboratory Methods
5.1. Single RT-PCR and PCR Assays
PCR amplifies DNA by repeated cycles of denaturation, annealing, and extension; RT-
PCR first converts RNA into cDNA and then amplifies it; single RT-PCR performs reverse
transcription and PCR in one tube for faster, less contaminated detection [71]. Single RT-
PCR successfully detects SPFMV , SPCSV , SPVG, IVMV , and SPLCV directly from in

In [20]:
splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_large_all = splitter_large.split_documents(all_documents)
print(f"Large chunks (1000 chars): {len(chunks_large_all)} chunks")



Large chunks (1000 chars): 3564 chunks


In [26]:
# the following was from AI to sanitize the metadata so weaviate would work

def sanitize_metadata(metadata: dict) -> dict:
    clean = {}
    for key, value in metadata.items():
        new_key = re.sub(r'[^_0-9A-Za-z]', '_', key)
        # Weaviate also requires the first char to be a letter or underscore
        if new_key[0].isdigit():
            new_key = f"_{new_key}"
        clean[new_key] = value
    return clean

In [27]:
chunks_for_gold = [
    Document(page_content=c.page_content, metadata=sanitize_metadata(c.metadata))
    for c in chunks_large_all
]

In [28]:
# Chat model through the UNC AI Gateway
llm = ChatOpenAI(
    model=CHAT_MODEL, # 4.1 mini is default 
    base_url=UNC_AI_BASE_URL,
    api_key=UNC_AI_API_KEY,
    temperature=0,
)

In [30]:
def file_type(chunk):
    ft = chunk.metadata.get("file_type")
    if ft:
        return ft
    ext = chunk.metadata.get("source", "").rsplit(".", 1)[-1].lower()
    return "html" if ext in ("htm", "html") else ext

MIN_LEN = {"csv": 100, "pdf": 500, "html": 400, "json":500 }
per_type = {"html": 0, "csv": 0, "pdf": 25, "json":25}   # only one definition

def filled_fields(chunk):
    """Return {column: value} for non-empty fields only."""
    out = {}
    for line in chunk.page_content.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            if v.strip():
                out[k.strip()] = v.strip()
    return out

MIN_FILLED = 4   # tune after looking at your data

def long_enough(c):
    text = c.page_content
    ft = file_type(c)
    if ft == "csv":
        return len(filled_fields(c)) >= MIN_FILLED
    if len(text) <= MIN_LEN.get(ft, 500):
        return False
    if ft == "html" and looks_like_code(text):
        return False
    if dupe_counts[norm(text)] > 1:
        return False
    return True
    
def long_enough(c):
    return len(c.page_content) > MIN_LEN.get(file_type(c), 500)

by_type = defaultdict(list)
for c in chunks_for_gold:
    if long_enough(c):
        by_type[file_type(c)].append(c)
print({k: len(v) for k, v in by_type.items()})

sampled = []
for ftype, n in per_type.items():
    pool = by_type.get(ftype, [])
    if not pool:
        print(f"WARNING: no eligible chunks for '{ftype}'")
    sampled += random.sample(pool, min(n, len(pool)))
print("sampled:", len(sampled))

def csv_passage(chunk):
    return "\n".join(f"{k}: {v}" for k, v in filled_fields(chunk).items())
    
class GoldQA(BaseModel):
    query: str = Field(description="A self-contained question answerable from the passage")
    reference_answer: str = Field(description="The correct answer, based only on the passage")

structured_llm = llm.with_structured_output(GoldQA)

prompt = """You are writing evaluation questions for a RAG system.

Read the passage below and write ONE question that can be answered using only this passage.

Rules:
- The question must make sense on its own. Never say "according to the passage/text/document".
- Don't copy phrasing from the passage; paraphrase so retrieval isn't trivially keyword-matched.
- Ask about a specific fact, explanation, or relationship, not something vague.
- The reference answer must be fully supported by the passage.

Passage:
{passage}
"""

csv_prompt = """Write ONE question for a RAG evaluation set that can be answered
from the record below alone. The person asking has never seen the data and will not be able to see the row
as part of the question.

- Identify the record using one or two of its values (for example a name or clone number).
- Ask about one other value in the record, in natural wording rather than column labels.
- Do not use the words entry, row, record, table, column, or dataset.

Record:
{passage}
"""

def gold_record(chunk, qa):
    m = chunk.metadata
    return {
        "query": qa.query,
        "reference_answer": qa.reference_answer,
        "file_type": file_type(chunk),
        "source": m.get("source"),
        "page": m.get("page"),            # pdf
        "row": m.get("row"),              # csv
        "section": m.get("section"),      # htm
        "section_idx": m.get("section_idx"),
        "start_index": m.get("start_index"),
        "chunk_id": m.get("chunk_id"),

        # Wikipedia metadata
        "title": m.get("title"),
        "source_url": m.get("source_url"),
        "revision_id": m.get("revision_id"),
        
        "source_chunk": chunk.page_content,
        "origin": "generated",
    }

generated = []
for chunk in sampled:
    p = csv_prompt if file_type(chunk) == "csv" else prompt
    qa = structured_llm.invoke(p.format(passage=chunk.page_content))
    generated.append(gold_record(chunk, qa))

{'pdf': 2898, 'json': 194}
sampled: 50


In [31]:
for i, g in enumerate(generated):
    print(f"--- {i} ---")
    print("FILE:", g["source"], f"({g['file_type']})")

    # show whichever location field exists for this file type
    if g.get("page") is not None:
        print("PAGE:", g["page"])
    if g.get("row") is not None:
        print("ROW:", g["row"])

    print("Q:", g["query"])
    print("A:", g["reference_answer"])
    print("CHUNK:", g["source_chunk"][:800], "\n")

--- 0 ---
FILE: corpus\new_files\proquest\Effect_of_fertilizer_on_bioche.pdf (pdf)
PAGE: 5
Q: How did the water content of sweet potato flour vary between the purple color skin and light cream color skin varieties under different fertilization treatments?
A: The water content of sweet potato flour varied by variety and treatment. For the purple color skin (PCS) variety, water content ranged from 70.09 ± 1.02% to 72.95 ± 2.98%, and for the light cream color skin (LCCS) variety, it ranged from 70.26 ± 6.26% to 72.26 ± 0.59%. The highest water content for both varieties was recorded under NPK fertilization (PCS: 72.95 ± 2.98%; LCCS: 72.26 ± 0.59%), followed by eluate treatment. The lowest water content was consistently observed with the digestate-compost mixture treatment (PCS: 70.09 ± 1.02%; LCCS: 70.26 ± 6.26%).
CHUNK: triplicate.
Statistical analysis
The ANOVA shows significant difference (p ˂ 0.05) or 
significance level of accepting the hypothesis was anal -
ysed. A study of 49 corre

In [32]:
 # indices from the printed "--- i ---" headers
keep = [0, 1, 2, 3, 5, 10, 11, 12, 16, 17, 18, 21, # new pdfs
        25, 26, 27, 28, 29, 30, 32, 35, 37, 39, 40, 41, 46, 47, 48, 49
       ]


In [47]:
LOC_KEY = {"pdf": "page", "csv": "row", "html": "section_idx", "json": "chunk_id"}

def to_gold(g):
    loc_key = LOC_KEY.get(g["file_type"])

    expected = {
        "source": Path(g["source"]).name
    }

    if loc_key and g.get(loc_key) is not None:
        expected[loc_key] = g[loc_key]

    return {
        "query": g["query"],
        "expected_sources": [expected],
        "reference_answer": g["reference_answer"],
        "origin": "generated",
    }



In [36]:
with open("gold_set.json", "r", encoding="utf-8") as f:
    gold_dataset = json.load(f)

print("Original gold questions:", len(gold_dataset))

Original gold questions: 34


In [37]:
import copy

gold_dataset_m2 = copy.deepcopy(gold_dataset)

new_items = [to_gold(generated[i]) for i in keep]

gold_dataset_m2.extend(new_items)

print("Original questions:", len(gold_dataset))
print("New questions:", len(new_items))
print("Milestone 2 total:", len(gold_dataset_m2))

Original questions: 34
New questions: 28
Milestone 2 total: 62


In [42]:
unanswerable_questions = [
    {
        "query": "What will be the leading cause of sweet potato crop losses in North Carolina in 2035?",
        "expected_sources": [],
        "reference_answer": "The available documents do not provide enough information to answer this question.",
        "origin": "unanswerable",
        "answerable": False
    },
    {
        "query": "How many acres of sweet potatoes were harvested in Siler City, North Carolina, in 2025?",
        "expected_sources": [],
        "reference_answer": "The available documents do not provide enough information to answer this question.",
        "origin": "unanswerable",
        "answerable": False
    },
    {
        "query": "Which sweet potato cultivar generated the highest net profit for North Carolina farmers in 2024?",
        "expected_sources": [],
        "reference_answer": "The available documents do not provide enough information to answer this question.",
        "origin": "unanswerable",
        "answerable": False
    },
    {
        "query": "Which sweet potato researcher has received the most international research awards as of 2026?",
        "expected_sources": [],
        "reference_answer": "The available documents do not provide enough information to answer this question.",
        "origin": "unanswerable",
        "answerable": False
    },
    {
        "query": "What percentage increase in sweet potato yield resulted from applying a newly developed fertilizer called SP-GrowthMax in controlled field trials?",
        "expected_sources": [],
        "reference_answer": "The available documents do not provide enough information to answer this question.",
        "origin": "unanswerable",
        "answerable": False
    },
    {
    "query": "What was the average improvement in sweet potato shelf life when stored under blue LED lighting compared with conventional dark storage?",
    "expected_sources": [],
    "reference_answer": "The available documents do not provide enough information to answer this question.",
    "origin": "unanswerable",
    "answerable": False
    }
]

In [43]:
gold_dataset_m2.extend(unanswerable_questions)

print("Total gold questions:", len(gold_dataset_m2))

Total gold questions: 73


In [44]:
unique_questions = []
seen = set()

for item in gold_dataset_m2:
    question = item["query"].strip().lower()

    if question not in seen:
        unique_questions.append(item)
        seen.add(question)

gold_dataset_m2 = unique_questions

print("Total unique questions:", len(gold_dataset_m2))

Total unique questions: 68


In [48]:
for item in gold_dataset_m2:
    for source in item["expected_sources"]:
        if source["source"].endswith(".json"):
            source.pop("chunk_id", None)

print("Updated Wikipedia gold references")

Updated Wikipedia gold references


In [52]:
for item in gold_dataset_m2:
    for source in item["expected_sources"]:
        if source["source"].endswith(".json"):
            source.pop("chunk_id", None)

In [54]:
with open("gold_set_m2.json", "w", encoding="utf-8") as f:
    json.dump(gold_dataset_m2, f, indent=4, ensure_ascii=False)

In [51]:
from collections import Counter

print("Total:", len(gold_dataset_m2))
print("By origin:", Counter(q.get("origin", "manual") for q in gold_dataset_m2))
print("Unanswerable:", sum(
    q.get("answerable") is False
    for q in gold_dataset_m2
))

Total: 68
By origin: Counter({'generated': 52, 'manual': 10, 'unanswerable': 6})
Unanswerable: 6
